# 2. ETF Screening and Selection

## Overview
This notebook implements the ETF screening and selection process across four asset classes:

1. **Equities** — distributing UCITS ETFs; filtered by platform availability, TER, dividends, size, and risk-adjusted returns
2. **Bonds** — distributing UCITS bond ETFs; same screening pipeline as equities with a wider Sharpe sensitivity (±0.25)
3. **Precious Metals** — physical gold/silver/platinum ETCs; accumulating allowed; screened globally by size and TER; hard-filtered to InvestEngine-available ETCs
4. **Commodities** — broad commodity ETCs (covering energy, agriculture, and metals); accumulating allowed; screened globally by size and TER; hard-filtered to InvestEngine-available ETCs

Each asset class produces a ranked shortlist saved to `data/intermediate/` (CSV backup) and to the `screened_etfs` table in the SQLite database.

## Setup and Data Loading

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path("..").resolve()))

import os
import json
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import requests
from scipy import stats

from etf_utils.config import DATA_RAW, DATA_INTERMEDIATE, DATA_OUTPUT, DATA_CONFIG
from etf_utils.data_provider import DataProvider
from etf_utils.data_io import get_region_category_from_filename, get_asset_class_from_filename
from etf_utils.metrics import calculate_annualized_volatility
from etf_utils.platform_check import check_etf_availability
from etf_utils.database import save_screened_etfs

provider = DataProvider()

## Platform Availability Check
First, we'll define a function to check if ETFs are available on our chosen platform (InvestEngine).

> **Note:** Platform availability filtering applies to **all four asset classes**. For equities and bonds it runs after the screening loop; for precious metals and commodities it is applied before the final `head(2)` selection so that unavailable ETCs are never included in the portfolio.

In [ ]:
# Calculate benchmark returns and volatility for 2025
benchmark_year_start = "2025-01-01"
benchmark_year_end = "2025-12-31"

eq_benchmark_symbol  = "VEVE"   # VEVE.L on yfinance
bnd_benchmark_symbol = "SAAA"   # SAAA.L on yfinance
pm_benchmark_symbol  = "SGLN"   # iShares Physical Gold ETC
cmd_benchmark_symbol = "CMOP"   # Invesco Bloomberg Commodity ETC

eq_prices  = provider.get_historical_prices(eq_benchmark_symbol,  benchmark_year_start, benchmark_year_end)
bnd_prices = provider.get_historical_prices(bnd_benchmark_symbol, benchmark_year_start, benchmark_year_end)
pm_prices  = provider.get_historical_prices(pm_benchmark_symbol,  benchmark_year_start, benchmark_year_end)
cmd_prices = provider.get_historical_prices(cmd_benchmark_symbol, benchmark_year_start, benchmark_year_end)

def _period_metrics(prices, start, end):
    yr = prices.loc[start:end]["close"]
    ret  = round((yr.iloc[-1] - yr.iloc[0]) / yr.iloc[0] * 100, 2)
    vol  = round(calculate_annualized_volatility(yr) * 100, 2)
    sr   = round(ret / vol, 2)
    return ret, vol, sr

eq_benchmark_return,  eq_benchmark_volatility,  eq_sharpe_ratio  = _period_metrics(eq_prices,  benchmark_year_start, benchmark_year_end)
bnd_benchmark_return, bnd_benchmark_volatility, bond_sharpe_ratio = _period_metrics(bnd_prices, benchmark_year_start, benchmark_year_end)
pm_benchmark_return,  pm_benchmark_volatility,  pm_sharpe_ratio  = _period_metrics(pm_prices,  benchmark_year_start, benchmark_year_end)
cmd_benchmark_return, cmd_benchmark_volatility, cmd_sharpe_ratio = _period_metrics(cmd_prices, benchmark_year_start, benchmark_year_end)

print(f"2025 VEVE  return: {eq_benchmark_return}%,  vol: {eq_benchmark_volatility}%,  Sharpe: {eq_sharpe_ratio}")
print(f"2025 SAAA  return: {bnd_benchmark_return}%, vol: {bnd_benchmark_volatility}%, Sharpe: {bond_sharpe_ratio}")
print(f"2025 SGLN  return: {pm_benchmark_return}%,  vol: {pm_benchmark_volatility}%,  Sharpe: {pm_sharpe_ratio}")
print(f"2025 CMOP  return: {cmd_benchmark_return}%, vol: {cmd_benchmark_volatility}%, Sharpe: {cmd_sharpe_ratio}")

In [4]:
benchmark_etfs = [
    # Structure: [Asset Class, Region, Country, Primary Ticker, Description]
    
    # Equity Benchmarks - Developed Markets
    ['Equity', 'Developed_AmericasandUK', 'United Kingdom', 'ISF.L', 'iShares Core FTSE 100 UCITS ETF GBP (Dist)'],
    ['Equity', 'Developed_AmericasandUK', 'United States', 'SPY', 'SPDR S&P 500 ETF Trust'],
    ['Equity', 'Developed_AmericasandUK', 'Canada', 'ZCN.TO', 'BMO S&P/TSX Capped Composite'],    
    ['Equity', 'Developed_EMEA', 'Germany', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'France', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Italy', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Spain', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Netherlands', 'CS51.L', 'iShares VII PLC - iShares Core EURO STOXX 50 ETF EUR Acc'],
    ['Equity', 'Developed_EMEA', 'Switzerland', 'CSWG.L', 'Amundi Index Solutions - Amundi MSCI Switzerland'],    
    ['Equity', 'Developed_APAC', 'Japan', 'XDJP.L', 'Xtrackers Nikkei 225 UCITS ETF 1D'],
    ['Equity', 'Developed_APAC', 'Australia', 'SAUS.L', 'iShares MSCI Australia UCITS ETF'],    
    # Equity Benchmarks - Emerging Markets
    ['Equity', 'Emerging_Americas', 'Brazil', 'IBZL.L', 'iShares MSCI Brazil UCITS ETF'],
    ['Equity', 'Emerging_Americas', 'Mexico', 'XMEX.L', 'iShares MSCI Mexico Capped ETF'],    
    ['Equity', 'Emerging_APACandEMEA', 'China', 'ASHR', 'XTRACKERS HARVEST CSI 300 CHINA A-SHARES ETF'],
    ['Equity', 'Emerging_APACandEMEA', 'India', 'XNIF.L', 'Xtrackers Nifty 50 Swap UCITS ETF 1C'],
    ['Equity', 'Emerging_APACandEMEA', 'South Korea', 'EWY', 'iShares MSCI South Korea ETF'],    
]

benchmark_df = pd.DataFrame(
    benchmark_etfs, 
    columns=['asset_class', 'region', 'country', 'benchmark_ticker', 'benchmark_description']
)

# Sort the DataFrame
benchmark_df = benchmark_df.sort_values(['asset_class', 'region', 'country'])

# Calculate 2025 returns for each benchmark ETF using DataProvider
benchmark_df['benchmark_2025_Return'] = benchmark_df['benchmark_ticker'].apply(
    lambda sym: round(provider.get_benchmark_period_return(sym, "2025-01-01", "2025-12-31") * 100, 2)
)
benchmark_df

,asset_class,region,country,benchmark_ticker,benchmark_description,benchmark_2025_Return
10,Equity,Developed_APAC,Australia,SAUS.L,iShares MSCI Australia UCITS ETF,4.64
9,Equity,Developed_APAC,Japan,XDJP.L,Xtrackers Nikkei 225 UCITS ETF 1D,18.58
2,Equity,Developed_AmericasandUK,Canada,ZCN.TO,BMO S&P/TSX Capped Composite,31.24
0,Equity,Developed_AmericasandUK,United Kingdom,ISF.L,iShares Core FTSE 100 UCITS ETF GBP (Dist),20.65
1,Equity,Developed_AmericasandUK,United States,SPY,SPDR S&P 500 ETF Trust,18.89
4,Equity,Developed_EMEA,France,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.38
3,Equity,Developed_EMEA,Germany,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.38
5,Equity,Developed_EMEA,Italy,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.38
7,Equity,Developed_EMEA,Netherlands,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.38
6,Equity,Developed_EMEA,Spain,CS51.L,iShares VII PLC - iShares Core EURO STOXX 50 E...,28.38


### Fetch benchmark close prices

Retrieve the previous trading day's close price for each of the four benchmark ETFs:

| Asset Class | Benchmark | Ticker |
|---|---|---|
| Equities | Vanguard FTSE Dev World | VEVE.L |
| Bonds | SPDR Bloomberg 0–3Y US Agg | SAAA.L |
| Precious Metals | iShares Physical Gold ETC | SGLN.L |
| Commodities | Invesco Bloomberg Commodity | CMOP.L |

## ETF Screening Process

Screen and rank ETFs within each asset class using pre-determined weights.

### Equities & Bonds
Filtered for distributing ETFs, platform availability, minimum AUM, and maximum TER. Ranked by a weighted composite of 1-year (50%), 3-year (30%), and 5-year (20%) risk-adjusted return/risk scores, adjusted by Sharpe ratio relative to the asset-class benchmark.

### Precious Metals
Screened globally. Accumulating ETCs permitted. Filtered by minimum AUM (>100 M), maximum TER (<0.60%), not hedged.

**Within-metal beta filter:** each ETC's 2025 return is compared against the **median** of its metal group (threshold 0.93). Uses median rather than max to avoid EUR/GBP currency outliers inflating the reference. This removes commodity-style ETPs (e.g. XGLD, BULP, SLVR) that consistently underperform physically-backed peers, without penalising cross-metal comparisons.

**Overlap-aware selection:** the chosen commodity ETC (CMOP) already tracks the Bloomberg Commodity Index, which contains Gold (14.29%) and Silver (4.49%). Selecting a gold ETC for PM would duplicate that exposure. ETCs are therefore sorted by BCOM overlap **ascending** — platinum (0%) and palladium (0%) are preferred over silver (4.49%), which is preferred over gold (14.29%).

**Metal diversity:** after the overlap sort, the screener picks the best ETC *per metal type* before choosing the top 2. This prevents selecting two ETCs backed by the same metal (e.g. two platinum funds) and ensures each selected ETC adds distinct exposure.

**Platform filter:** available on InvestEngine (hard filter applied before final selection).

### Commodities
Screened globally. Accumulating ETCs permitted. Filtered by minimum AUM (>100 M), maximum TER (<0.60%), not hedged.

**Beta filter (≥ 1 vs. 2025 CMOP benchmark):** ETCs must have matched or beaten the CMOP benchmark return in 2025. This removes underperformers (e.g. ENCG 0%, UC15 1.58%) before the InvestEngine check.

**Platform filter:** pure single-commodity ETCs (crude oil, wheat, etc.) are not listed on InvestEngine; the hard filter therefore naturally selects only diversified broad-commodity ETCs (CMOP, COMM, COMX, BCOG etc.).

In [6]:
# get_region_category_from_filename and get_asset_class_from_filename
# are imported from etf_utils.data_io

main_df_equities = pd.DataFrame()

for filepath in sorted(DATA_RAW.glob("justetf_class-equity*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)

        if not asset_class or region_category == 'Unknown' or asset_class != 'equity':
            print(f'Skipping {csvl}')
            continue

        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')

        distributing_df = csv_df.copy()
        distributing_df = distributing_df[distributing_df['dividends'] == 'Distributing']
        distributing_df = distributing_df[distributing_df['size'] > 100]
        distributing_df = distributing_df[distributing_df['hedged'] == False]

        include_tickers = ['IBZL']
        distributing_df = distributing_df[
            (distributing_df['ter'] < 0.5) | (distributing_df['ticker'].isin(include_tickers))
        ]

        # Tickers to exclude (re-evaluate with yfinance â€” some may now be available)
        remove_tickers = ['XUCN', 'LYXIB', 'C001', 'SW2CHA', 'XSMI', 'SJPD', 'XESD', 'C024']
        distributing_df = distributing_df[~distributing_df['ticker'].isin(remove_tickers)]

        hy_df = (distributing_df.copy()
                 .sort_values(by=['last_year_dividends'], ascending=False)
                 .drop_duplicates(subset=['region_category']))
        hy_df['intra_region_category'] = 'High Yield'

        benchmark_distributing_df = distributing_df.copy()[
            ~distributing_df['country'].isin(hy_df['country'])
        ]
        benchmark_distributing_df = pd.merge(
            benchmark_distributing_df,
            benchmark_df[['country', 'benchmark_ticker', 'benchmark_description', 'benchmark_2025_Return']],
            on='country', how='left'
        )
        benchmark_distributing_df["beta"] = (
            benchmark_distributing_df["2025"] / benchmark_distributing_df["benchmark_2025_Return"]
        )

        # Save intermediate benchmark data
        intermediate_path = DATA_INTERMEDIATE / f'benchmark_distributing_df_{csvl}'
        benchmark_distributing_df.to_csv(intermediate_path, index=False)

        benchmark_distributing_df = benchmark_distributing_df[benchmark_distributing_df['beta'] >= 1]
        beta_df = (benchmark_distributing_df.copy()
                   .sort_values(by=['ter', 'beta'], ascending=[True, False])
                   .drop_duplicates(subset=['region_category']))
        beta_df['intra_region_category'] = 'Beta'

        main_df_equities = pd.concat([main_df_equities, hy_df, beta_df], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

# Check InvestEngine availability
main_df_equities['available_on_investengine'] = (
    main_df_equities['ticker'].apply(check_etf_availability)
)

# Fetch latest close price via DataProvider (yfinance, no API key)
def _get_price(ticker):
    try:
        return provider.get_latest_price(ticker)
    except Exception as e:
        print(f"Price error for {ticker}: {e}")
        return None, None

if not main_df_equities.empty:
    main_df_equities[['yday_close_price_date', 'yday_close_price']] = (
        main_df_equities['ticker'].apply(_get_price).to_list()
    )
else:
    main_df_equities['yday_close_price_date'] = []
    main_df_equities['yday_close_price'] = []

main_df_equities.to_csv(DATA_INTERMEDIATE / 'summary_equities.csv', index=False)
save_screened_etfs(main_df_equities, 'equity', portfolio_year=2026)
print(f"Saved {len(main_df_equities)} equity ETFs")

Processing equity file for developed_americasanduk: justetf_class-equity_developed_americasanduk.csv
Processing equity file for developed_apac: justetf_class-equity_developed_apac.csv
Processing equity file for developed_emea: justetf_class-equity_developed_emea.csv


Processing equity file for emerging_americas: justetf_class-equity_emerging_americas.csv
Processing equity file for emerging_apacandemea: justetf_class-equity_emerging_apacandemea.csv


Saved 5 equity ETFs


In [7]:
############# BONDS #############
main_df_bonds = pd.DataFrame()

# Manually curated bond ETF list
bonds_data = {
    'asset_class': ['bonds'] * 8,
    'region_category': [
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_AmericasandUK', 'Developed_AmericasandUK',
        'Developed_EMEA', 'Developed_EMEA', 'Developed_EMEA',
        'Emerging_APACandEMEA',
    ],
    'intra_region_category': ['Govt', 'Corp', 'Govt', 'Corp', 'Govt', 'Corp', 'High yield', 'Corp'],
    'ticker': ['IGLT', 'SLXX', 'TRXG', 'UC81', 'PRIR', 'VECP', 'JNKE', 'EMCP'],
}
df_bonds_list = pd.DataFrame(bonds_data)
df_bonds_list = df_bonds_list[df_bonds_list['ticker'] != 'JNKE']  # not on InvestEngine

for filepath in sorted(DATA_RAW.glob("justetf_class-bonds_*.csv")):
    csvl = filepath.name
    try:
        asset_class = get_asset_class_from_filename(csvl)
        region_category = get_region_category_from_filename(csvl)
        print(f'Processing {asset_class} file for {region_category}: {csvl}')

        try:
            csv_df = pd.read_csv(filepath)
            if csv_df.empty:
                print(f'Empty file: {csvl}')
                continue
        except pd.errors.EmptyDataError:
            print(f'Empty file: {csvl}')
            continue

        csv_df['asset_class'] = asset_class
        csv_df['region'] = csv_df['region'].fillna('N/A')
        csv_df['country'] = csv_df['country'].fillna('N/A')

        for _, row in df_bonds_list.iterrows():
            if row['ticker'] in csv_df['ticker'].values:
                ticker = row['ticker']
                print(f'  Found {ticker} in {csvl}')
                csv_df_ticker = csv_df[csv_df['ticker'] == ticker].copy()
                csv_df_ticker['intra_region_category'] = row['intra_region_category']
                csv_df_ticker['region_category'] = row['region_category']
                main_df_bonds = pd.concat([main_df_bonds, csv_df_ticker], ignore_index=True)

    except Exception as e:
        print(f'Error processing {csvl}: {e}')

main_df_bonds['available_on_investengine'] = (
    main_df_bonds['ticker'].apply(check_etf_availability)
)

if not main_df_bonds.empty:
    main_df_bonds[['yday_close_price_date', 'yday_close_price']] = (
        main_df_bonds['ticker'].apply(_get_price).to_list()
    )
else:
    main_df_bonds['yday_close_price_date'] = []
    main_df_bonds['yday_close_price'] = []

main_df_bonds.to_csv(DATA_INTERMEDIATE / 'summary_bonds.csv', index=False)
save_screened_etfs(main_df_bonds, 'bonds', portfolio_year=2026)
print(f"Saved {len(main_df_bonds)} bond ETFs")

Processing bonds file for developed_americasanduk: justetf_class-bonds_developed_americasanduk.csv
  Found IGLT in justetf_class-bonds_developed_americasanduk.csv
  Found SLXX in justetf_class-bonds_developed_americasanduk.csv
  Found TRXG in justetf_class-bonds_developed_americasanduk.csv
  Found UC81 in justetf_class-bonds_developed_americasanduk.csv
  Found EMCP in justetf_class-bonds_developed_americasanduk.csv
Processing bonds file for developed_apac: justetf_class-bonds_developed_apac.csv
Processing bonds file for developed_emea: justetf_class-bonds_developed_emea.csv
  Found PRIR in justetf_class-bonds_developed_emea.csv
  Found VECP in justetf_class-bonds_developed_emea.csv
Processing bonds file for emerging_americas: justetf_class-bonds_emerging_americas.csv
Empty file: justetf_class-bonds_emerging_americas.csv
Processing bonds file for emerging_apacandemea: justetf_class-bonds_emerging_apacandemea.csv


Saved 7 bond ETFs


In [ ]:
############# PRECIOUS METALS #############
# Relaxed filters: allow accumulating ETCs, higher TER ceiling, no distributing requirement.
#
# Overlap-aware selection: the chosen commodity ETC (CMOP/COMM) already tracks the Bloomberg
# Commodity Index, which includes Gold (14.29%) and Silver (4.49%).  Selecting a gold ETC for
# the PM allocation would duplicate ~14% of the commodity position.  We therefore sort by
# commodity_overlap_pct ASC so that platinum/palladium (0% in BCOM) are preferred over
# silver, and silver is preferred over gold.
#
# Metal diversity: after the overlap sort we pick the best ETC *per metal type* and then
# select the top 2 types.  This prevents the screener from choosing two platinum (or two
# palladium) ETCs when both happen to be cheapest — ensuring each selected ETC gives
# exposure to a different metal.
#
# Within-metal beta filter: removes ETCs whose 2025 return significantly lags the median
# of their metal group (threshold 0.93).  Uses group median (not max) to avoid EUR/GBP
# currency outliers inflating the reference.  Catches commodity-style ETPs (e.g. XGLD,
# BULP, SLVR) that underperform physically-backed peers.
#
# BCOM weights are read from data/config/commodity_index_weights.json (update annually).

import json as _json

_bcom = _json.loads((DATA_CONFIG / 'commodity_index_weights.json').read_text())
_pm_bcom = _bcom['precious_metals']   # {'gold': 14.29, 'silver': 4.49, 'platinum': 0.0, ...}


def _pm_commodity_overlap(name: str) -> float:
    """Return the BCOM overlap % for a PM ETC based on its underlying metal.

    Platinum and palladium are absent from BCOM → 0% overlap (preferred).
    Silver → 4.49%, Gold → 14.29% (deprioritised — already in commodity ETC).
    Mixed precious-metals baskets → full PM group weight (18.78%).
    """
    n = name.lower()
    if 'palladium' in n:
        return _pm_bcom['palladium']
    if 'platinum' in n:
        return _pm_bcom['platinum']
    if 'silver' in n:
        return _pm_bcom['silver']
    if 'gold' in n:
        return _pm_bcom['gold']
    # Mixed basket (e.g. "WisdomTree Physical Precious Metals")
    return _pm_bcom['mixed']


def _metal_type(name: str) -> str:
    """Classify a PM ETC by its underlying metal for diversity-aware selection."""
    n = name.lower()
    if 'palladium' in n:
        return 'palladium'
    if 'platinum' in n:
        return 'platinum'
    if 'silver' in n:
        return 'silver'
    if 'gold' in n:
        return 'gold'
    return 'mixed'


pm_raw_path = DATA_RAW / 'justetf_class-preciousMetals_global.csv'
main_df_pm = pd.DataFrame()

if pm_raw_path.exists():
    pm_df = pd.read_csv(pm_raw_path)
    pm_df['asset_class'] = 'preciousMetals'
    pm_df['region_category'] = 'Global'
    pm_df['intra_region_category'] = 'Global'
    pm_df['region'] = pm_df.get('region', 'Global')
    pm_df['country'] = pm_df.get('country', 'Global')

    pm_filtered = pm_df[
        (pm_df['size'] > 100) &
        (pm_df['ter'] < 0.6) &
        (pm_df['hedged'] == False)
    ].copy()

    # Tag each ETC with its commodity-index overlap % and metal type
    pm_filtered['commodity_overlap_pct'] = pm_filtered['name'].apply(_pm_commodity_overlap)
    pm_filtered['metal_type'] = pm_filtered['name'].apply(_metal_type)

    # Within-metal beta filter: remove ETCs that significantly lag their metal-peer group.
    # Uses group median (not max) to avoid outliers from currency/domicile differences.
    # Threshold 0.93 catches commodity-style ETPs (XGLD, BULP, SLVR) that underperform
    # physically-backed peers without penalising cross-metal comparisons.
    pm_filtered_beta = pm_filtered[pm_filtered['2025'].notna()].copy()
    if not pm_filtered_beta.empty:
        metal_group_median = pm_filtered_beta.groupby('metal_type')['2025'].transform('median')
        pm_filtered_beta['within_metal_beta'] = pm_filtered_beta['2025'] / metal_group_median
        pm_filtered = pm_filtered_beta[pm_filtered_beta['within_metal_beta'] >= 0.93]
    else:
        pm_filtered['within_metal_beta'] = None

    # Sort: least overlap first (prefer platinum/palladium), then cheapest TER, then best return/risk
    pm_filtered = pm_filtered.sort_values(
        ['commodity_overlap_pct', 'ter', 'last_year_return_per_risk'],
        ascending=[True, True, False]
    )

    # Filter to InvestEngine-available ETCs before selecting top 2
    pm_filtered['available_on_investengine'] = pm_filtered['ticker'].apply(check_etf_availability)
    pm_filtered = pm_filtered[pm_filtered['available_on_investengine'] == True]

    # Metal diversity: pick the best-ranked ETC within each metal type, then take top 2 types.
    # pm_filtered is already sorted correctly; groupby first() preserves that order within groups.
    best_per_metal = (
        pm_filtered
        .groupby('metal_type', sort=False)
        .first()
        .reset_index()
        .sort_values(['commodity_overlap_pct', 'ter'], ascending=[True, True])
    )
    main_df_pm = best_per_metal.head(2).copy()

    if not main_df_pm.empty:
        main_df_pm[['yday_close_price_date', 'yday_close_price']] = (
            main_df_pm['ticker'].apply(_get_price).to_list()
        )
    else:
        main_df_pm['yday_close_price_date'] = []
        main_df_pm['yday_close_price'] = []

    main_df_pm.to_csv(DATA_INTERMEDIATE / 'summary_preciousMetals.csv', index=False)
    save_screened_etfs(main_df_pm, 'preciousMetals', portfolio_year=2026)
    print(f"Saved {len(main_df_pm)} precious metals ETCs  "
          f"(BCOM year: {_bcom['year']})")
    display(main_df_pm[['ticker', 'name', 'metal_type', 'ter', 'dividends', 'size',
                         '2025', 'within_metal_beta', 'last_year_return_per_risk',
                         'commodity_overlap_pct', 'available_on_investengine', 'yday_close_price']])
else:
    print(f"Warning: {pm_raw_path} not found — run notebook 01 first to scrape precious metals data.")

############# COMMODITIES #############
# Broad commodity ETCs covering energy, agriculture and metals.
#
# Beta filter: ETC must match or beat the CMOP benchmark 2025 return (beta >= 1).
# This removes underperformers (e.g. ENCG 0%, UC15 1.58%) before the InvestEngine check.
#
# Platform filter: pure single-commodity ETCs (crude oil, wheat, etc.) are not listed on
# InvestEngine; the hard filter therefore naturally selects only diversified broad ETCs.

cmd_raw_path = DATA_RAW / 'justetf_class-commodities_global.csv'
main_df_cmd = pd.DataFrame()

if cmd_raw_path.exists():
    cmd_df = pd.read_csv(cmd_raw_path)
    cmd_df['asset_class'] = 'commodities'
    cmd_df['region_category'] = 'Global'
    cmd_df['intra_region_category'] = 'Global'
    cmd_df['region'] = cmd_df.get('region', 'Global')
    cmd_df['country'] = cmd_df.get('country', 'Global')

    cmd_filtered = cmd_df[
        (cmd_df['size'] > 100) &
        (cmd_df['ter'] < 0.6) &
        (cmd_df['hedged'] == False)
    ].sort_values(['ter', 'last_year_return_per_risk'], ascending=[True, False]).copy()

    # Beta filter: keep only ETCs that matched or beat the CMOP benchmark return in 2025
    cmd_filtered = cmd_filtered[cmd_filtered['2025'].notna()].copy()
    if cmd_benchmark_return and cmd_benchmark_return > 0:
        cmd_filtered['beta'] = cmd_filtered['2025'] / cmd_benchmark_return
        cmd_filtered = cmd_filtered[cmd_filtered['beta'] >= 1]
    else:
        cmd_filtered['beta'] = None  # benchmark return unavailable — skip filter

    # Hard filter: only InvestEngine-available ETCs (this naturally keeps broad ETCs)
    cmd_filtered['available_on_investengine'] = cmd_filtered['ticker'].apply(check_etf_availability)
    cmd_filtered = cmd_filtered[cmd_filtered['available_on_investengine'] == True]

    main_df_cmd = cmd_filtered.head(2).copy()

    if not main_df_cmd.empty:
        main_df_cmd[['yday_close_price_date', 'yday_close_price']] = (
            main_df_cmd['ticker'].apply(_get_price).to_list()
        )
    else:
        main_df_cmd['yday_close_price_date'] = []
        main_df_cmd['yday_close_price'] = []

    main_df_cmd.to_csv(DATA_INTERMEDIATE / 'summary_commodities.csv', index=False)
    save_screened_etfs(main_df_cmd, 'commodities', portfolio_year=2026)
    print(f"Saved {len(main_df_cmd)} commodity ETCs")
    display(main_df_cmd[['ticker', 'name', 'ter', 'dividends', 'size', 'last_year_return_per_risk',
                          '2025', 'beta', 'available_on_investengine', 'yday_close_price']])
else:
    print(f"Warning: {cmd_raw_path} not found — run notebook 01 first to scrape commodities data.")

############# COMBINED SUMMARY #############
summary_df = pd.concat(
    [df for df in [main_df_equities, main_df_bonds, main_df_pm, main_df_cmd] if not df.empty],
    ignore_index=True,
)
summary_df.to_csv(DATA_INTERMEDIATE / 'summary_all.csv', index=False)
print(f"Saved {len(summary_df)} total ETFs to summary_all.csv")
summary_df[['asset_class', 'region_category', 'intra_region_category', 'ticker', 'name',
            'ter', 'available_on_investengine']]